# Unsloth QLoRA Fine-Tuning Demo
**Model:** `unsloth/Phi-3-mini-4k-instruct-bnb-4bit`  
**Method:** QLoRA (Quantized Low-Rank Adaptation) + Supervised Fine-Tuning (SFT)

## What this notebook does
1. Loads a 4-bit quantized Phi-3-mini model using Unsloth (fast & memory-efficient)
2. Attaches small trainable LoRA adapters (only ~3% of params are trained)
3. Fine-tunes on a tiny dataset: given a sentence about a person, extract name/age/job as JSON
4. Runs inference and measures accuracy on held-out test cases

## Why QLoRA?
- **4-bit quantization** cuts VRAM usage to ~6 GB, making this runnable on a laptop GPU
- **LoRA** adds tiny adapter matrices to the frozen base model — fast to train, easy to swap out
- **Unsloth** provides 2× faster training with the same results through kernel optimizations

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available: ", torch.cuda.is_available())
print("GPU count:      ", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:       ", torch.cuda.get_device_name(0))
    print("VRAM (GB):      ", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## Step 1 — Load the Base Model
We load Phi-3-mini in 4-bit (NF4) format. Unsloth handles quantization automatically.

In [ ]:
from unsloth import FastLanguageModel

# Maximum number of tokens the model can see at once.
# 2048 covers nearly all examples in this dataset.
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    # Pre-quantized 4-bit checkpoint hosted by Unsloth on HuggingFace.
    model_name="unsloth/Phi-3-mini-4k-instruct-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    # dtype=None lets Unsloth auto-detect the best dtype for your GPU
    # (bfloat16 on Ampere+, float16 otherwise).
    dtype=None,
    # load_in_4bit=True is the key flag for QLoRA — saves ~4× VRAM.
    load_in_4bit=True,
)

print("Model loaded! Parameters:", sum(p.numel() for p in model.parameters()) / 1e9, "B")

## Step 2 — Attach LoRA Adapters
Instead of updating all 3.8 B parameters, LoRA injects tiny trainable matrices at selected layers.
Only ~3% of parameters are trained, but the model learns the new task effectively.

Key hyperparameters:
- **`r` (rank)**: Controls adapter size. Higher = more capacity but more VRAM. 64 is a good default.
- **`lora_alpha`**: Scaling factor. Convention: set to `2 * r`.
- **`target_modules`**: Which projection layers to apply LoRA to. Covering all attention + MLP layers gives best results.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    # LoRA rank: higher rank = more expressive adapters, but uses more memory.
    r=64,
    # Apply LoRA to all attention projection layers (q, k, v, o)
    # and both MLP layers (gate, up, down) for maximum coverage.
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    # Scaling factor: convention is 2× rank.
    lora_alpha=128,
    # No dropout during fine-tuning — the dataset is small so we want maximum learning.
    lora_dropout=0,
    # Freeze all bias terms; only the LoRA matrices are trainable.
    bias="none",
    # Unsloth's custom gradient checkpointing: reduces peak VRAM at almost no speed cost.
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable/1e6:.1f}M / {total/1e9:.2f}B ({100*trainable/total:.2f}%)")

## Step 3 — Prepare the Dataset
We use `people_data.json` — 20 examples of the form:
```
User:  "At midnight, Mira, aged 29, works as a cybersecurity analyst."
Model: {"name": "Mira", "age": "29", "job": "cybersecurity analyst"}
```
Each example is converted into Phi-3's chat template format, then used as plain text for SFT.

In [ ]:
import json
from datasets import Dataset

# Load the 20 training examples from a local JSON file.
with open("people_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

raw_dataset = Dataset.from_list(data)
print(f"Loaded {len(raw_dataset)} training examples")
print("Keys:", raw_dataset.column_names)

In [ ]:
def to_chat_text(example):
    """
    Format one training example as a Phi-3 chat string.

    Phi-3's chat template wraps messages like:
        <|user|>\n...text...<|end|>\n<|assistant|>\n...text...<|end|><|endoftext|>

    SFTTrainer trains the model to produce everything after <|assistant|>.
    """
    response = example["response"]

    # If the response is a dict/list (structured JSON), serialize it to a string.
    if not isinstance(response, str):
        response = json.dumps(response, ensure_ascii=False)

    messages = [
        {"role": "user",      "content": example["prompt"]},
        {"role": "assistant", "content": response},
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,            # Return a plain string, not token IDs.
            add_generation_prompt=False,  # Don't add the assistant turn opener.
        )
    }

# Apply the formatter to every example; drop the original columns.
dataset = raw_dataset.map(to_chat_text, remove_columns=raw_dataset.column_names)

# Preview the first example so we can confirm the format looks right.
print(dataset[0]["text"])

## Step 4 — Fine-Tune with SFTTrainer
SFT (Supervised Fine-Tuning) trains the model by maximizing the likelihood of the correct response.

Training budget:
- **60 steps** × effective batch size 8 = 480 gradient updates worth of signal from 20 examples
- Takes ~60 s on an RTX 5070 Ti

If you want to experiment, try increasing `max_steps` to 120 or `r` to 128.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    # The column that contains our formatted chat strings.
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=SFTConfig(
        # --- Batch size & gradient accumulation ---
        # 2 samples per GPU, accumulated over 4 steps → effective batch = 8.
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,

        # --- Learning rate schedule ---
        learning_rate=2e-4,   # Standard LoRA learning rate.
        warmup_steps=10,      # Linearly ramp up LR for the first 10 steps.

        # --- Training duration ---
        max_steps=60,         # Stop after 60 optimizer steps (overrides num_train_epochs).
        num_train_epochs=3,   # Fallback epoch limit (max_steps takes priority).

        # --- Optimizer ---
        # 8-bit AdamW: same quality as full AdamW but uses ~4× less VRAM for optimizer states.
        optim="adamw_8bit",

        # --- Logging & saving ---
        logging_steps=10,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer.train()

## Step 5 — Inference
We switch the model to inference mode (Unsloth enables faster kernel paths) and test on 36 held-out examples.

In [ ]:
import torch

# Switch to inference mode: Unsloth swaps in faster forward-pass kernels.
FastLanguageModel.for_inference(model)

# Clear the generation_config's max_length so we can use max_new_tokens
# without triggering the "both max_new_tokens and max_length are set" warning.
model.generation_config.max_length = None

def predict(prompt: str, max_new_tokens: int = 128) -> str:
    """
    Run inference on a single prompt and return the model's reply.

    Args:
        prompt: The user message (plain text).
        max_new_tokens: Max tokens the model may generate.

    Returns:
        The model's response as a stripped string.
    """
    messages = [{"role": "user", "content": prompt}]

    # Build the Phi-3 chat prompt and convert to GPU tensors.
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,  # Appends <|assistant|>\n so the model starts its reply.
        return_tensors="pt",
    ).to("cuda")

    # Build an explicit attention mask (1 = attend, 0 = ignore).
    # Passing it avoids the "attention_mask not set" warning.
    attention_mask = torch.ones_like(input_ids)

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        # Greedy-ish decoding: low temperature keeps output deterministic.
        temperature=0.1,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

    # Slice off the input tokens — we only want what the model generated.
    new_tokens = outputs[:, input_ids.shape[1]:]
    return tokenizer.decode(new_tokens[0], skip_special_tokens=True).strip()


print("Inference helper ready.")

## Step 6 — Evaluate on Test Set
Run the model on 36 unseen examples and check whether each JSON response is an exact match.

In [ ]:
# Load the 36 held-out test examples.
with open("people_test_data.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

correct = 0

for idx, example in enumerate(test_data, 1):
    response = predict(example["prompt"])

    # Try to parse the response as JSON and compare to the ground truth.
    try:
        predicted = json.loads(response)
        match = predicted == example["expected_response"]
    except json.JSONDecodeError:
        # The model produced malformed JSON — count as incorrect.
        match = False

    if match:
        correct += 1

    # Print every example for inspection.
    status = "✓" if match else "✗"
    print(f"[{status}] #{idx:02d}")
    print(f"  Prompt:    {example['prompt'][:80]}...")
    print(f"  Expected:  {json.dumps(example['expected_response'], ensure_ascii=False)}")
    print(f"  Predicted: {response}")
    print()

accuracy = correct / len(test_data) * 100
print("=" * 60)
print(f"Accuracy: {accuracy:.1f}%  ({correct}/{len(test_data)} correct)")